In [1]:
import scanpy as sc
import cellxgene_census
import cellxgene_census.experimental as czi_exp
import tiledb
import os
import pandas as pd
import time
import yaml

# Set up

In [2]:
# Verify that the proxy settings are correctly set
http_proxy = os.getenv('http_proxy')
https_proxy = os.getenv('https_proxy')

print("HTTP_PROXY: {}".format(http_proxy))
print("HTTPS_PROXY: {}".format(https_proxy))

um_proxy = "proxy1.arc-ts.umich.edu" ##### this is the setup used below in the open_soma call
um_proxy_port = "3128"

config= {
    "vfs.s3.proxy_host": um_proxy, 
    "vfs.s3.proxy_port": um_proxy_port,
    "vfs.s3.request_timeout_ms": 100000000, 
}

cellxgene_census.get_census_version_description('stable')

HTTP_PROXY: http://proxy1.arc-ts.umich.edu:3128/
HTTPS_PROXY: http://proxy1.arc-ts.umich.edu:3128/


{'release_date': None,
 'release_build': '2025-01-30',
 'soma': {'uri': 's3://cellxgene-census-public-us-west-2/cell-census/2025-01-30/soma/',
  'relative_uri': '/cell-census/2025-01-30/soma/',
  's3_region': 'us-west-2'},
 'h5ads': {'uri': 's3://cellxgene-census-public-us-west-2/cell-census/2025-01-30/h5ads/',
  'relative_uri': '/cell-census/2025-01-30/h5ads/',
  's3_region': 'us-west-2'},
 'flags': {'lts': True}}

In [3]:
with cellxgene_census.open_soma(census_version="2024-07-01", tiledb_config=config) as census: 
    var = cellxgene_census.get_var(
        census = census,
        organism = "Homo sapiens",
    )
    census.close()

var.head()

,soma_joinid,feature_id,feature_name,feature_length,nnz,n_measured_obs
0,0,ENSG00000000003,TSPAN6,4530,4530448,73855064
1,1,ENSG00000000005,TNMD,1476,236059,61201828
2,2,ENSG00000000419,DPM1,9276,17576462,74159149
3,3,ENSG00000000457,SCYL3,6883,9117322,73988868
4,4,ENSG00000000460,C1orf112,5970,6287794,73636201


In [4]:
CENSUS_VERSION = "2024-07-01"

for e in czi_exp.get_all_available_embeddings(CENSUS_VERSION):
    print(f"{e['embedding_name']:15} {e['experiment_name']:15} {e['data_type']:15}")

geneformer      homo_sapiens    obs_embedding  
scvi            mus_musculus    obs_embedding  
scvi            homo_sapiens    obs_embedding  
scgpt           homo_sapiens    obs_embedding  


# Load the cell type spec

In [5]:
fpath = "../resources/hsc_cell_types.yaml"
cell_dict = yaml.safe_load(open(fpath, 'r'))
cell_dict.keys()



dict_keys(['hsc_and_progenitors', 'myeloid_cells', 'lymphoid_cells', 'innate_lymphoid_cells', 'fibroblasts', 'mesenchymal_cells', 'endothelial_cells'])

In [6]:
# Initialize TileDB configuration
config= {
    "vfs.s3.proxy_host": um_proxy, 
    "vfs.s3.proxy_port": um_proxy_port,
    "vfs.s3.request_timeout_ms": 1000000, 
}

# --- Census Opening ---
print("Opening cellxgene census...")
open_census_start_time = time.time()

census = cellxgene_census.open_soma(
    census_version="2024-07-01",  # Consider using "latest" or a specific date if needed
    tiledb_config=config,
)

open_census_end_time = time.time()
open_census_duration = open_census_end_time - open_census_start_time
print(f"Census opened successfully in {open_census_duration:.2f} seconds.")

Opening cellxgene census...
Census opened successfully in 5.47 seconds.


# Extract the data 

In [7]:
import time
import scanpy as sc
import cellxgene_census

outdir = "/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/"

# --- Data Extraction and Processing ---
for data_label, cell_types in cell_dict.items():
    print(f"\nProcessing data for label: {data_label} (cell types: {', '.join(cell_types)})")

    # --- Get AnnData ---
    get_anndata_start_time = time.time()
    print("  Starting AnnData retrieval...")

    # Build the cell type filter string
    cell_type_filter_parts = [f"cell_type == '{cell_type}'" for cell_type in cell_types]
    cell_type_filter = " or ".join(cell_type_filter_parts)

    # Construct the complete obs_value_filter
    obs_value_filter = f"is_primary_data == True and disease == 'normal' and ({cell_type_filter})"

    # Retrieve AnnData object
    try:
        adata = cellxgene_census.get_anndata(
            census=census,
            organism="Homo sapiens",
            obs_value_filter=obs_value_filter,
        )
    except Exception as e:
        print(f"  Error retrieving AnnData: {e}")
        continue  # Skip to the next iteration if an error occurs

    get_anndata_end_time = time.time()
    get_anndata_duration = get_anndata_end_time - get_anndata_start_time
    print(f"  Retrieved AnnData in {get_anndata_duration:.2f} seconds.")


    # --- Memory Usage ---
    print("  AnnData shape:", adata.shape)
    print("  Memory usage before writing:")
    sc.logging.print_memory_usage()

    # --- Write AnnData to file ---
    write_anndata_start_time = time.time()
    print("  Starting AnnData write...")

    outpath = f"{outdir}{data_label}.h5ad"
    try:
        adata.write(outpath)
        print(f"  Successfully wrote AnnData to {outpath}")
    except Exception as e:
        print(f"  Error writing AnnData to file: {e}")

    write_anndata_end_time = time.time()
    write_anndata_duration = write_anndata_end_time - write_anndata_start_time
    print(f"  Wrote AnnData in {write_anndata_duration:.2f} seconds.")

    print("  Memory usage after writing:")
    sc.logging.print_memory_usage()


print("\nData processing complete.")


Processing data for label: hsc_and_progenitors (cell types: hematopoietic stem cell, hematopoietic multipotent progenitor cell, lymphoid lineage restricted progenitor cell, early lymphoid progenitor, common myeloid progenitor, granulocyte monocyte progenitor cell, megakaryocyte-erythroid progenitor cell, erythroid progenitor cell, megakaryocyte progenitor cell, basophil mast progenitor cell, common dendritic progenitor)
  Starting AnnData retrieval...
  Retrieved AnnData in 158.63 seconds.
  AnnData shape: (66394, 60530)
  Memory usage before writing:
Memory usage: current 4.23 GB, difference +4.23 GB
  Starting AnnData write...
  Successfully wrote AnnData to /nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/hsc_and_progenitors.h5ad
  Wrote AnnData in 6.93 seconds.
  Memory usage after writing:
Memory usage: current 4.25 GB, difference +0.02 GB

Processing data for label: myeloid_cells (cell types: myeloid cell, myelocyte, promyelocyte (early and late), neutrophil

In [8]:
# outdir = "/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/"

# # --- Data Extraction and Processing ---
# for data_label, cell_types in cell_dict.items():
#     print(f"\nProcessing data for label: {data_label} (cell types: {', '.join(cell_types)})")
#     get_anndata_start_time = time.time()

#     # Build the cell type filter string
#     cell_type_filter_parts = [f"cell_type == '{cell_type}'" for cell_type in cell_types]
#     cell_type_filter = " or ".join(cell_type_filter_parts)

#     # Construct the complete obs_value_filter
#     obs_value_filter = f"is_primary_data == True and disease == 'normal' and ({cell_type_filter})"

#     # Retrieve AnnData object
#     try:
#         adata = cellxgene_census.get_anndata(
#             census=census,
#             organism="Homo sapiens",
#             obs_value_filter=obs_value_filter,
#         )
#     except Exception as e:
#         print(f"  Error retrieving AnnData: {e}")
#         continue  # Skip to the next iteration if an error occurs

#     get_anndata_end_time = time.time()
#     get_anndata_duration = get_anndata_end_time - get_anndata_start_time
#     print(f"  Retrieved AnnData in {get_anndata_duration:.2f} seconds.")

#     # Memory usage
#     print(adata.shape)
#     print("  Current memory usage:")
#     sc.logging.print_memory_usage()

#     # Write AnnData to file
#     outpath = f"{outdir}{data_label}.h5ad"
#     try:
#         adata.write(outpath)
#         print(f"  Successfully wrote AnnData to {outpath}")
#     except Exception as e:
#         print(f"  Error writing AnnData to file: {e}")

# print("\nData processing complete.")